[Reference](https://medium.com/@dlab-berkeley/generating-embeddings-from-a-self-supervised-speech-model-025b6c93f77d$0)

In [2]:
# standard libraries for data science
import pandas as pd
import numpy as np
from pathlib import Path
# tools to download our dataset
import urllib.request
import zipfile
# specialized libraries for audio and .wav processing
import phonlab as pb
import librosa
# tools to convert directories to dataframes
import fileorganize
# progress bars
from tqdm.auto import tqdm
# machine learning packages
from transformers import AutoFeatureExtractor, AutoModel
import torch
# tools for our visualization
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

In [3]:
# automatically select the best hardware accelerator if available, otherwise fallback to CPU
device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
print(f"Using device: {device}")

In [4]:
# download the dataset if you don't have it already
# this can take a few minutes
if not Path("AudioMNIST").exists():
    with tqdm(unit="B", unit_scale=True, desc="Downloading AudioMNIST") as bar:
        def hook(blocks, block_size, total_size):
            if total_size > 0:
                bar.total = total_size
            bar.update(block_size)
        urllib.request.urlretrieve(
            "https://github.com/soerenab/AudioMNIST/archive/refs/heads/master.zip",
            "AudioMNIST.zip",
            reporthook=hook,
        )
    with zipfile.ZipFile("AudioMNIST.zip") as z:
        for name in tqdm(z.namelist(), desc="Extracting"):
            z.extract(name)
    Path("AudioMNIST-master").rename("AudioMNIST")
    Path("AudioMNIST.zip").unlink()
# save the data directory as a constant
DATA_DIR = Path("AudioMNIST/data/")

# create dataframe of files
dir_df = fileorganize.dir_to_df(DATA_DIR)
# remove metadata file(s) that aren't wavs
wav_mask = dir_df["fname"].str.endswith('.wav')
dir_df = dir_df.loc[wav_mask, :].reset_index(drop=True)
# select 500 random recordings
dir_df = (pd.concat([dir_df, dir_df['fname']
    .str.extract(r"^(?P<digit>\d+)_(?P<speaker>\d+)_(?P<token>\d+).wav")], axis='columns')
    .sample(n=500, random_state=77)
    .reset_index(drop=True))

relpath, fname, digit, speaker, token = dir_df.loc[0, :]

MODEL = "microsoft/wavlm-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL).to(device).eval()

# model requires 16000Hz sampling rate
target_sr = 16000
# acquire filepath to wav file
fpath = DATA_DIR / relpath / fname
# load audio and sampling rate (for confirmation)
audio_array, sampling_rate = pb.loadsig(fpath, fs=target_sr)

# create inputs
inputs = feature_extractor(audio_array, sampling_rate = sampling_rate, return_tensors = "pt")
# move inputs to chip
inputs = {k: v.to(device) for k, v in inputs.items()}
# run model to generate outputs
with torch.inference_mode():
    outputs = model(**inputs, output_hidden_states = True)

def extract_features(audio_array, audio_sr, layer_idx):
    # create inputs
    inputs = feature_extractor(audio_array, sampling_rate = audio_sr, return_tensors = "pt")
    # move inputs to chip
    inputs = {k: v.to(device) for k, v in inputs.items()}
    # run model to generate outputs
    with torch.inference_mode():
        outputs = model(**inputs, output_hidden_states = True)
    # extract a specific layer
    layer_features = outputs.hidden_states[layer_idx]
    return layer_features.squeeze(0).cpu().numpy()

embedding_df = []
for idx, relpath, fname, digit, speaker, token in tqdm(dir_df.itertuples(), total=len(dir_df)):
    target_sr = 16000
    fpath = DATA_DIR / relpath / fname
    audio_array, sampling_rate = pb.loadsig(fpath, fs=target_sr)
    embedding = extract_features(audio_array, sampling_rate, -1)
    embedding_df.append({
        "audio_arr": audio_array,
        "sampling_rate": sampling_rate,
        "embedding": embedding,
    })

df = pd.concat([dir_df, pd.DataFrame(embedding_df)], axis='columns')

def statpool(x: np.ndarray) -> np.ndarray:
    assert x.ndim == 2
    means = x.mean(axis=0)
    sds = x.std(axis=0) + 1e-5
    return np.concatenate([means, sds])
df['statpool'] = df['embedding'].map(statpool)

# use pca to reducee the statpooled embeddings to three components
# store them in the dataframe
df[["x_coord", "y_coord", "z_coord"]] = PCA(n_components=3).fit_transform(np.stack(df['statpool']))

mfccs_statpooled = []
for idx, audio_arr, sampling_rate in df[['audio_arr', 'sampling_rate']].itertuples():
    mfccs = librosa.feature.mfcc(y=audio_arr, sr=sampling_rate, n_mfcc=20, n_fft=400, hop_length=160).T
    mfccs_statpooled.append(statpool(mfccs))
mfcc_features = np.stack(mfccs_statpooled)
mfcc_features = StandardScaler().fit_transform(mfcc_features)
mfcc_coords = PCA(n_components=3).fit_transform(mfcc_features)
mfcc_df = df[["digit"]].copy()
mfcc_df[["x_coord", "y_coord", "z_coord"]] = mfcc_coords